# Liquidity Hedge Protocol — Integration Test Analysis

Analysis of real-world integration test results from the off-chain protocol emulator.
Data collected from a live 30-minute test on Solana mainnet using a real Orca SOL/USDC concentrated liquidity position.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import json
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = Path('data')

## 1. Load Data

In [ ]:
# Monitor timeline (one row per 60s snapshot)
monitor = pd.read_csv(DATA_DIR / 'monitor-timeline.csv')
monitor['timestamp_utc'] = pd.to_datetime(monitor['timestamp_utc'])
monitor['elapsed_min'] = monitor['elapsed_s'] / 60
print(f'Monitor snapshots: {len(monitor)} rows over {monitor.elapsed_s.max()/60:.0f} minutes')
monitor.head()

In [ ]:
# Simulated payout curve
payouts = pd.read_csv(DATA_DIR / 'simulated-payouts.csv')
print(f'Simulated price levels: {len(payouts)}')
payouts

In [ ]:
# Performance summary (single row)
summary = pd.read_csv(DATA_DIR / 'performance-summary.csv')
s = summary.iloc[0]
print('=== Test Summary ===')
print(f'Entry price:      ${s.entry_price:.4f}')
print(f'Settlement price: ${s.settlement_price:.4f}')
print(f'Price change:     {s.price_change_pct:+.4f}%')
print(f'Position PnL:     ${s.pos_pnl_usd:+.6f}')
print(f'IL:               {s.il_pct:.4f}%')
print(f'Premium:          ${s.premium_usd:.6f}')
print(f'Payout:           ${s.payout_usd:.6f}')
print(f'Cert outcome:     {s.cert_outcome}')
print(f'Hedged PnL:       ${s.hedged_pnl_usd:+.6f}')
print(f'RT return:        {s.rt_return_pct:+.4f}%')
print(f'Gas cost:         ${s.gas_usd:.4f}')

## 2. Price & Position Value Over Time

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price
ax1.plot(monitor['elapsed_min'], monitor['sol_price'], 'b-', linewidth=1.5, label='SOL Price')
ax1.axhline(y=s.entry_price, color='green', linestyle='--', alpha=0.7, label=f'Entry ${s.entry_price:.2f}')
ax1.axhline(y=s.barrier_price, color='red', linestyle='--', alpha=0.7, label=f'Barrier ${s.barrier_price:.2f}')
ax1.set_ylabel('SOL Price ($)')
ax1.legend(loc='upper right')
ax1.set_title('SOL Price During Test Period')

# Position value vs hold value
ax2.plot(monitor['elapsed_min'], monitor['pos_value_usd'], 'b-', linewidth=1.5, label='Position Value')
ax2.plot(monitor['elapsed_min'], monitor['hold_value_usd'], 'g--', linewidth=1.5, label='Hold Value')
ax2.set_xlabel('Elapsed (minutes)')
ax2.set_ylabel('Value ($)')
ax2.legend(loc='upper right')
ax2.set_title('Position Value vs Hold Value (IL = gap between lines)')

plt.tight_layout()
plt.savefig(DATA_DIR / 'price_and_value.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Impermanent Loss Over Time

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(monitor['elapsed_min'], monitor['il_usd'] * 1e6, 'r-', linewidth=1.5)
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax1.set_ylabel('IL (micro-USD)')
ax1.set_title('Impermanent Loss Over Time (micro-USD scale)')

ax2.plot(monitor['elapsed_min'], monitor['il_pct'], 'r-', linewidth=1.5)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax2.set_xlabel('Elapsed (minutes)')
ax2.set_ylabel('IL (%)')
ax2.set_title('Impermanent Loss Over Time (%)')

plt.tight_layout()
plt.savefig(DATA_DIR / 'impermanent_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Simulated Payout Curve (Corridor Product)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Payout vs price change
ax = axes[0, 0]
ax.bar(payouts['change_pct'], payouts['payout_usdc'], width=0.8,
       color=['#d32f2f' if b else '#1976d2' for b in payouts['barrier_breached']])
ax.axvline(x=0, color='green', linestyle='--', alpha=0.7, label='Entry')
barrier_pct = (s.barrier_price - s.entry_price) / s.entry_price * 100
ax.axvline(x=barrier_pct, color='red', linestyle='--', alpha=0.7, label=f'Barrier ({barrier_pct:.1f}%)')
ax.set_xlabel('Price Change (%)')
ax.set_ylabel('Payout (USDC)')
ax.set_title('Certificate Payout by Price Change')
ax.legend()

# LP Net PnL (hedged) vs price change
ax = axes[0, 1]
colors = ['#4caf50' if v >= 0 else '#d32f2f' for v in payouts['lp_net_pnl_usd']]
ax.bar(payouts['change_pct'], payouts['lp_net_pnl_usd'], width=0.8, color=colors)
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.axvline(x=barrier_pct, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Price Change (%)')
ax.set_ylabel('LP Net PnL ($)')
ax.set_title('LP Hedged PnL by Price Change')

# RT PnL vs price change
ax = axes[1, 0]
colors = ['#4caf50' if v >= 0 else '#d32f2f' for v in payouts['rt_pnl_usd']]
ax.bar(payouts['change_pct'], payouts['rt_pnl_usd'], width=0.8, color=colors)
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.axvline(x=barrier_pct, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Price Change (%)')
ax.set_ylabel('RT PnL ($)')
ax.set_title('RT (Risk Taker) PnL by Price Change')

# CL Loss vs Payout overlay
ax = axes[1, 1]
ax.plot(payouts['change_pct'], payouts['cl_loss_usdc'], 'b-o', markersize=4, label='CL Position Loss')
ax.plot(payouts['change_pct'], payouts['payout_usdc'], 'r-s', markersize=4, label='Payout')
ax.axvline(x=0, color='green', linestyle='--', alpha=0.5, label='Entry')
ax.axvline(x=barrier_pct, color='red', linestyle='--', alpha=0.5, label='Barrier')
ax.set_xlabel('Price Change (%)')
ax.set_ylabel('Amount (USDC)')
ax.set_title('CL Position Loss vs Corridor Payout')
ax.legend()

plt.tight_layout()
plt.savefig(DATA_DIR / 'payout_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Protocol Economics Summary

In [ ]:
premium = s.premium_usd
payout = s.payout_usd
cap = s.cap_usdc

print('=== Protocol Economics ===')
print(f'Premium collected:     ${premium:.6f}')
print(f'Payout disbursed:      ${payout:.6f}')
print(f'Net to pool (RT):      ${premium - payout:+.6f}')
print(f'LP cost of hedge:      ${premium - payout:.6f}')
print(f'Cap utilization:       ${payout/cap*100:.4f}% of ${cap:.2f} cap') if cap > 0 else None
print(f'Pool utilization:      {s.util_after_bps/100:.2f}%')
print(f'Gas overhead:          ${s.gas_usd:.4f}')
print()

print('=== Premium Breakdown ===')
# These come from the performance CSV — extract if available
print(f'Total premium:         ${premium:.6f}')
print(f'As % of cap:           {premium/cap*100:.4f}%')
print(f'As % of position:      {premium/s.pos_entry_usd*100:.4f}%')
print()

print('=== Risk/Return Profile ===')
print(f'RT return (30 min):    {s.rt_return_pct:+.6f}%')
print(f'RT annualized:         {s.rt_return_pct * (365*24*2):+.2f}% (extrapolated)')
print(f'LP position PnL:       ${s.pos_pnl_usd:+.8f}')
print(f'LP hedged PnL:         ${s.hedged_pnl_usd:+.8f}')
print(f'Hedge effectiveness:   {"PAYOUT matched loss" if payout > 0 and abs(payout - abs(s.pos_pnl_usd)) < 0.01 else "Certificate expired (price above entry)"}')

## 6. IL Correlation with Price Movement

In [ ]:
# Compute price change from entry
entry_price = s.entry_price
monitor['price_change_pct'] = (monitor['sol_price'] - entry_price) / entry_price * 100

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(monitor['price_change_pct'], monitor['il_pct'],
                     c=monitor['elapsed_min'], cmap='viridis', s=30, alpha=0.8)
plt.colorbar(scatter, label='Elapsed (min)')
ax.set_xlabel('Price Change from Entry (%)')
ax.set_ylabel('Impermanent Loss (%)')
ax.set_title('IL vs Price Change (colored by time)')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='green', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(DATA_DIR / 'il_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Audit Log Analysis

In [ ]:
# Parse audit log
audit_entries = []
with open(DATA_DIR / 'audit.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            audit_entries.append(json.loads(line))

print(f'Audit entries: {len(audit_entries)}')
print()
for entry in audit_entries:
    ts = entry.get('timestamp', 'N/A')
    op = entry.get('operation', 'N/A')
    result = entry.get('result', 'N/A')
    params = entry.get('params', {})
    print(f'  [{ts}] {op} -> {result}')
    if 'premiumUsdc' in params:
        print(f'    premium: {params["premiumUsdc"]/1e6:.6f} USDC, cap: {params.get("capUsdc", 0)/1e6:.6f} USDC')
    if 'payout' in params:
        print(f'    payout: {params["payout"]/1e6:.6f} USDC, state: {params.get("state", "?")}')  
    if 'settlementPriceE6' in params:
        print(f'    settlement price: ${params["settlementPriceE6"]/1e6:.6f}')

## 8. Key Observations

Fill in after running the notebook with your data.

In [ ]:
print('=== Key Observations ===')
print()
print(f'1. Position size: ${s.pos_entry_usd:.4f} ({s.pos_entry_usd/entry_price:.6f} SOL equivalent)')
print(f'2. Premium as % of position: {premium/s.pos_entry_usd*100:.2f}%')
print(f'3. Price moved {s.price_change_pct:+.4f}% in 30 minutes')
print(f'4. Max IL observed: {monitor.il_pct.min():.6f}%')
print(f'5. IL range: [{monitor.il_pct.min():.6f}%, {monitor.il_pct.max():.6f}%]')
print(f'6. Position was in-range: {monitor.in_range.all()}')
print(f'7. Total gas cost: ${s.gas_usd:.4f} ({s.gas_usd/s.pos_entry_usd*100:.2f}% of position)')
print(f'8. Settlement outcome: {s.cert_outcome}')
if payout > 0:
    print(f'9. Payout covered {payout/abs(s.pos_pnl_usd)*100:.1f}% of position loss')
else:
    print(f'9. No payout (price ended above entry)')